In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import FileReadTool, TXTSearchTool, CSVSearchTool, ScrapeWebsiteTool, \
    SerperDevTool, EXASearchTool
from langchain_openai import ChatOpenAI
import os
from exa_py import Exa
from utils import get_openai_api_key, get_serper_api_key, get_exa_api_key

In [17]:
# Get api keys
openai_api_key = get_openai_api_key()
exa_api_key = get_exa_api_key()
serper_api_key = get_serper_api_key()

In [ ]:
serper_search = SerperDevTool()
exa_search = EXASearchTool()
scrape_tool = ScrapeWebsiteTool()
data_read_tool = FileReadTool(file_path='malaria_selected_countries.csv')
data_search_csv = CSVSearchTool(csv='./malaria_selected_countries.csv')

In [19]:
# Agent: Project Supervisor
project_supervisor = Agent(
    role = "Project Supervisor",
    goal = """Your goal is to"
        1. direct, review, guide and organize the academic research project.\
        2. ensure follow a logical and coherent structure \
        3. Relevant statistical analysis are provided for the research project that demonstrate\
            expertise and deep knowledge in statistical analysis techniques and statistical computing. \
        4. contents are relevant to the domain of available data""", 
    backstory = """You are a Professor of Statistics supervising an academic research project. You maintain \
        oversight of the project to meet the quality, content and formatting standards for the award of \
        Post Graduate Diploma in Statistics at the department of statistics in the faculty of science of a \
        leading, world-class, University.""",
    allow_delegation=True,
    tools = [data_read_tool, data_search_csv],
	verbose=True
)


In [20]:
# Agent: Public Health Specialist
malaria_specialist = Agent(
    role = "Malaria Specialist",
    goal = """Your goal is to ensure. \
        1. insightful, factual and accurate contents about Malaria to the research \
        2. cotents follow a logical and coherent structure that allign to industry standards for \
            target audience""",
    backstory = "You are a Public health professional specializing in Malaria in Africa. \
        You understand the dynamic of the desease, transmission, its impact on Africa and \
        initiatives to combat the desease and their effectiveness.",
    allow_delegation=True,
    tools = [data_read_tool, data_search_csv],
	verbose=True
)


In [21]:
#Agent: Content Planner
content_planner = Agent(
    role = "Content Planner",
    goal = "Plan engaging and factually accurate academic report content",
    backstory = "You're working on planning an academic research report on project topic"
        "You collaborate with Public Health Specialist and Project Supervisor to make informed decisions."
        "on the structure and outlines for the contents"
        "Your work is the basis for Content Researcher and the Content Writer.",
    allow_delegation=False,
    tools = [data_read_tool, data_search_csv],
	verbose=True
)

In [22]:
# Agent: Content Researcher
content_researcher = Agent(
    role = """Content Researcher""",
    goal = """Your goal is to: \
        1. research credible sources of insightful, accurate and factual information \
            including websites and publications from WHO, UNICEF, CDC, and other domain experts in \
            West Africa and specifically those in the selected countries in the dataset \
        2. collect and reference insightful, accurate and factual information, information from the \
            sources relevant to the {topic}.""",
    backstory = """You are a seasoned, resourceful and prolific content researcher for academic writing \
        You find the most relevant accurate and factual information \
        Your work is the basis for the work of Content Writer""",
    allow_delegation = False,
    tools = [exa_search],
    verbose = True
)


In [23]:
# Agent: Data Analyst
data_analyst = Agent(
    role = "Data Analyst",
    goal = "Your goal is to conduct statistical data analysis required and generate insightful visualization",
    backstory = "You generate R code for statistical analysis and data visualization that communicate \
        insights in accordance with the relevant statistical analysis required or as directed by Project Supervisor",
    allow_delegation=False,
    tools = [data_read_tool, data_search_csv],
	verbose=True
)


In [24]:
# Agent: Content Writer
content_writer = Agent(
    role = "Content Writer",
    goal = "Write content for academic research report, resolve gramatical and typographical errors",
    backstory = """
        1. You use the results from Content Researcher and Data Analyst to write clear, \
        concise and engaging academic research contents. \
        2. You avoid plagiarisim and use inline citation and referencing of sources. \
        3. Include R code for statistical analysis and visualization from Data Analyst\
        """,
    allow_delegation=False,
    tools = [data_read_tool, data_search_csv],
	verbose=True
)

In [25]:
# Task
definition = Task(
    description = """"Given Malaria data, complete the following task description:\
        1. define an academic research project topic \
        2. provide a problem statement \
        3. state the aims and objectives of the academic research project \
        4. state the scope of the project \
        5. state the methodology of the research \
        6. list the statistical analysis that will be required for the project \
        7. state the limitations of the project.\
        8. Provide references for sources of resources relevant to the project domain.""",
    expected_output="""A comprehensive report with the following outline \
        - project topic \
        - problem statement \
        - project aims and objective \
        - scope of the project \
        - research methodology \
        - list of relevant statistical analysis to be conducted \
        - limitation of the research project\
        - references""",
    tools = [exa_search, scrape_tool, data_read_tool, data_search_csv], 
    agent=content_writer
)

In [ ]:
# Crew
crew = Crew(
    agents=[project_supervisor, malaria_specialist, data_analyst, content_researcher],
    tasks=[definition],
    manager_llm=ChatOpenAI(model="gpt-4o", temperature=0.8), 
    process=Process.hierarchical,
    verbose=True, 
    memory=True
)


2024-12-09 21:03:31,909 - 5908 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [27]:
result = crew.kickoff()

# Agent: Crew Manager
## Task: "Given Malaria data, complete the following task description:        1. define an academic research project topic         2. provide a problem statement         3. state the aims and objectives of the academic research project         4. state the scope of the project         5. state the methodology of the research         6. list the statistical analysis that will be required for the project         7. state the limitations of the project.        8. Provide references for sources of resources relevant to the project domain.
# Agent: Content Writer
## Task: Complete a comprehensive report on malaria data, following the given outline: project topic, problem statement, project aims and objectives, scope of the project, research methodology, list of relevant statistical analysis, limitations of the research project, and references. Ensure the problem statement clearly defines the gap and research necessity and expand on the limitations section to include po

In [28]:
print(result)

### Comprehensive Report on Malaria Data

#### Project Topic
This research project focuses on the analysis of trends in malaria incidence across selected African countries from 2000 to 2022.

#### Problem Statement
Despite significant global efforts to reduce malaria incidence, the disease continues to pose a substantial public health challenge in many African countries. According to the World Health Organization (WHO, 2021), malaria remains a leading cause of morbidity and mortality across the continent. Previous studies indicate varying trends in malaria cases, underscoring the need for a comprehensive analysis to identify patterns, peak incidences, and the impact of interventions over time. This study aims to bridge the gap in understanding the nuances of malaria trends by providing a granular analysis of data covering two decades, focusing on selected countries where the burden is highest.

#### Project Aims and Objectives
The primary aim of this project is to explore trends in mal